# DLT Prototype V2

Let's do some performance benchmarking comparing to Orbit

## Library and Settings

In [ ]:
import pandas as pd
import numpy as np
import logging
import numpyro
numpyro.set_host_device_count(8)
numpyro.enable_x64(False)

import jax
import matplotlib.pyplot as plt

from wunkui.models.dlt import (
    run_dlt_model, 
    make_inference,
)

from wunkui.regression import (
    RegressionScheme,
    make_fourier_series_with_index, 
)

In [ ]:
%load_ext watermark
%watermark -v -n -m -p numpy,jax,numpyro,pandas,matplotlib,wunkui -u -n

In [ ]:
# Get or create logger
logger = logging.getLogger("wunkui")

# Set logger level
logger.setLevel(logging.DEBUG)

# Add a StreamHandler if no handlers are attached
if not logger.hasHandlers():
    handler = logging.StreamHandler()
    handler.setLevel(logging.DEBUG)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)

    logger.addHandler(handler)

# Example usage
logger.debug("This is a DEBUG message")
logger.info("This is an INFO message")

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# model_tag = "missing-values-seasonal-events"
model_tag = "seasonal-events"

## Data

In [ ]:
df = pd.read_csv(f"../resource/{model_tag}/df.csv", parse_dates=["date"])
train_df = df[:-100]
test_df = df[-100:]
saturation_df = pd.read_csv(f"../resource/{model_tag}/saturation_df.csv").set_index("regressor")
response = train_df["sales"].values
# response_norm = response.mean()
response_norm = 1.
y = np.log(response / response_norm)

resource_cols = [
    "promo", "radio", "search", "social", "tv"
]

# we should encode a miss-value signal
train_df.loc[:, resource_cols] = train_df.loc[:, resource_cols].fillna(0)

event_cols = [
    "new-years-day",
    "martin-luther-king-jr-day",
    "washingtons-birthday",
    "memorial-day",
    "independence-day",
    "labor-day",
    "columbus-day",
    "veterans-day",
    "thanksgiving-day",
    "christmas-day",
    "independence-day-observed",
    "juneteenth-national-independence-day-observed",
    "juneteenth-national-independence-day",
    "christmas-day-observed",
    "new-years-day-observed",
]
# ['independence-day', 'juneteenth-national-independence-day', 'labor-day', 'memorial-day', 'new-years-day', 'new-years-day-observed', 'thanksgiving-day', 'veterans-day', 'veterans-day-observed', 'washingtons-birthday']
event_cols = [x for x in event_cols if x in df.columns]

## Train model and data process

In [ ]:
rs = RegressionScheme()

In [ ]:
covariates = []
var_names = []

x_glb_intercept = np.ones((len(y), 1))
x_glb_slp = np.expand_dims(np.arange(len(y)) / 365.25, -1)
covariates += [x_glb_intercept, x_glb_slp]

var_names += ["glb_intercept", "glb_slp",]

rs.add_regressors(
    ["glb_intercept", "glb_slp"],
    loc_prior=[0., 0.],
    scale_prior=[10.0, 0.001],
)


if model_tag in ["seasonal-events", "seasonal-event-adstock", "missing-values-seasonal-events"]:
    x_annual_seas = make_fourier_series_with_index(len(y), period=365.25, order=3)
    x_weekly_seas = make_fourier_series_with_index(len(y), period=7, order=2)
    x_seas = np.concatenate((x_annual_seas, x_weekly_seas), axis=1)
    # x_seas = x_annual_seas
    rs.add_regressors(
        [f"seas_{i}" for i in range(x_seas.shape[1])],
        loc_prior=[0.] * x_seas.shape[1],
        scale_prior=[0.1] * x_seas.shape[1],
    )
    covariates.append(x_seas)
    var_names += [f"seas_{i}" for i in range(x_seas.shape[1])]

if model_tag in ["seasonal-events", "events", "missing-values-seasonal-events"]:
    x_event = train_df[event_cols].values
    covariates.append(x_event)
    coef_loc_event = [0.] * x_event.shape[1]
    coef_scale_event = [1.0] * x_event.shape[1]
    var_names += event_cols
    rs.add_regressors(
        event_cols,
        loc_prior=coef_loc_event,
        scale_prior=coef_scale_event,
    )

x_resource = train_df[resource_cols].values
sat_arr = saturation_df.loc[resource_cols, "saturation"].values
x_resource = np.log1p(x_resource / sat_arr)
covariates.append(x_resource)
var_names += resource_cols
rs.add_regressors(
    resource_cols,
    coef_sign="+",
    loc_prior=[0.] * x_resource.shape[1],
    scale_prior=[0.1] * x_resource.shape[1],
)

# stack all covariates and related input
covariates = np.concatenate(covariates, axis=1)
var_name = np.array(var_names)

train_covariates_df = pd.DataFrame(
    covariates, 
    columns=var_name, 
)

print(f"var_names: {var_name}")
print(f"var_name shape: {var_name.shape}")
print(f"covariates shape: {train_covariates_df.shape}")
print(f"covariates_df head: {train_covariates_df.head()}")

print(f"regression scheme: {rs.scheme}")

In [ ]:
# from our best model last time
# lev_sm=0.0266
lev_sm=0.00245
# lev_sm = 0.001
slp_sm=0.95
theta=0.94900
# theta=0.00

In [ ]:
rs.scheme

In [ ]:
seed = 2023
rng_key = jax.random.PRNGKey(seed)
print(f"rng_key: {rng_key}")
# split into 2 keys
rng_key, rng_key_mcmc = jax.random.split(rng_key)
print(f"rng_key: {rng_key}")
print(f"rng_key_mcmc: {rng_key_mcmc}")
posteriors = run_dlt_model(
    rng_key=rng_key_mcmc,
    y=y,
    lev_sm=lev_sm,
    slp_sm=slp_sm,
    theta=theta,
    # cauchy_sd=np.max(y[np.isfinite(y)])/30,
    regression_scheme=rs,
    covariates_df=train_covariates_df,
    mcmc_run_args={
        "num_warmup": 200,
        "num_samples": 125,
        "num_chains": 8,
        "progress_bar": True,
    },
)

## Diagnostics

In [ ]:
import arviz as az
summary_df = az.summary(
    posteriors,
    coords={"var_name": rs.scheme.index.to_numpy()},
    var_names=["coef", "sigma"],
    round_to=5,
)
summary_df

In [ ]:
dlt_comp = posteriors["dlt_comp"].to_numpy()
dlt_comp = dlt_comp.reshape(-1, *dlt_comp.shape[2:])
dlt_comp_mean = np.mean(np.exp(dlt_comp), axis=0)
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
ax.plot(dlt_comp_mean, label="level")

In [ ]:
coefs_df = pd.read_csv(f"../resource/{model_tag}/coefs_df.csv", index_col="regressor")
coefs_df

In [ ]:
coef_names = [f"coef[{x}]" for x in resource_cols]
coef_p50 = summary_df.loc[coef_names, "mean"].values
coef_true = coefs_df.loc[resource_cols, "coef"].values
print(f"Coef P50: {coef_p50}")
print(f"Coef True: {coef_true}")
mse_err = np.mean(
    np.abs(coef_true - coef_p50)
)

smape_err = np.mean(
    2 * np.abs(coef_true - coef_p50) / (np.abs(coef_true) + np.abs(coef_p50))
)

print(f"Coef SMAPE (%): {smape_err:.3%}")
print(f"MSE: {mse_err:.4f}")

## Forecast

### Covariates Input for future
Create covariates in forecast period

In [ ]:
covariates = []
n_steps = test_df.shape[0]
n_train_steps = train_df.shape[0]
x_glb_intercept = np.ones((n_steps, 1))
x_glb_slp = np.expand_dims(np.arange(n_train_steps, n_train_steps + n_steps) / 365.25, -1)
covariates += [x_glb_intercept, x_glb_slp]

if model_tag in ["seasonal-events", "seasonal-event-adstock", "missing-values-seasonal-events"]:
    x_annual_seas = make_fourier_series_with_index(n_steps, period=365.25, order=3, shift=n_train_steps)
    x_weekly_seas = make_fourier_series_with_index(n_steps, period=7, order=2, shift=n_train_steps)
    x_seas = np.concatenate((x_annual_seas, x_weekly_seas), axis=1)
    covariates.append(x_seas)

if model_tag in ["seasonal-events", "events", "missing-values-seasonal-events"]:
    x_event = test_df[event_cols].values
    covariates.append(x_event)

x_resource = test_df[resource_cols].values
sat_arr = saturation_df.loc[resource_cols, "saturation"].values
x_resource = np.log1p(x_resource / sat_arr)
covariates.append(x_resource)

# stack all covariates and related input
covariates = np.concatenate(covariates, axis=1)

future_covariates_df = pd.DataFrame(
    covariates, 
    columns=var_name, 
)

print(f"train_covariates_df shape: {train_covariates_df.shape}")
print(f"future_covariates_df shape: {future_covariates_df.shape}")
covariates_df = pd.concat(
    [train_covariates_df, future_covariates_df],
    ignore_index=True,
    axis=0,
)
print(f"full covariates_df shape: {covariates_df.shape}")

### Generate Forecast

In [ ]:
n_train_steps = train_df.shape[0]
forecast_horizon = test_df.shape[0]
# split into 2 keys
rng_key, rng_key_forecast = jax.random.split(rng_key)
forecast_samples = make_inference(
    rng_key=rng_key_forecast,
    posteriors=posteriors,
    lev_sm=lev_sm,
    slp_sm=slp_sm,
    theta=theta,
    covariates_df=covariates_df,
    transform_callback=lambda x: np.exp(x) * response_norm,
)

In [ ]:
yhat_lower, yhat_mid, yhat_upper = np.quantile(
    forecast_samples["forecast_samples"].values,
    q=[0.1, 0.5, 0.9], 
    axis=0
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# in-sample data plot
ax.scatter(df["date"].values, df["sales"].values, label='Observed', alpha=0.5, color="orange", s=10)
ax.plot(df["date"].values, yhat_mid, label='Forecast', alpha=0.8, color="dodgerblue")
ax.fill_between(df["date"].values, yhat_lower, yhat_upper, alpha=0.5, label='95% Prediction Interval', color="dodgerblue")

# vertical line indicate the forecast start
ax.axvline(test_df["date"].values[0], color='gray', linestyle='--', label='Forecast Start')

ax.set_title('Damped Local Trend Forecast')
